In [1]:
#################################################
# 1. Load Credentials from .env
#################################################

import os
from dotenv import load_dotenv
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd

load_dotenv()

USER = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASSWORD")

print("Credentials loaded.")

Credentials loaded.


In [2]:
#################################################
# 2. DBRepo Client Setup
#################################################

from dbrepo.RestClient import RestClient

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

client = RestClient(
    BASE_URL,
    username=USER,
    password=PASSWORD
)

print("DBRepo client initialized.")

DBRepo client initialized.


In [3]:
#################################################
# 3. Create / Get Database (robust version)
#################################################

DB_NAME = "data_stew_grp22_air_quality"
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"

databases = client.get_databases()
DB_ID = None

# Check if DB exists
for db in databases:
    if db.name == DB_NAME:
        DB_ID = db.id
        print("Using existing DB:", DB_ID)
        break

# Create if not exists
if DB_ID is None:
    print("Creating database...")

    try:
        client.create_database(
            DB_NAME,
            CONTAINER_ID,
            is_public=False,
            is_schema_public=False
        )
    except Exception:
        print("Create request sent (ignore validation error)")

    # Re-fetch databases
    databases = client.get_databases()

    for db in databases:
        if db.name == DB_NAME:
            DB_ID = db.id
            print("Database found after creation:", DB_ID)
            break

print("Final DB ID:", DB_ID)

Using existing DB: 81d82941-cae0-4cba-b27d-1bd883dd713a
Final DB ID: 81d82941-cae0-4cba-b27d-1bd883dd713a


In [4]:
#################################################
# 4. Clean Existing Tables
#################################################

tables = client.get_tables(DB_ID)

if len(tables) == 0:
    print("Database already clean.")
else:
    for t in tables:
        print("Deleting:", t.name)
        client.delete_table(DB_ID, t.id)

Deleting: t_measurement
Deleting: t_time
Deleting: t_station


In [5]:
#################################################
# 5. Create Table: t_station (safe)
#################################################

tables = client.get_tables(DB_ID)
table_names = [t.name for t in tables]

if "t_station" in table_names:
    print("t_station already exists. Skipping.")
else:
    df_station = pd.DataFrame({
        "station_id": [0],
        "station_name": ["dummy"],
        "latitude": [0.0],
        "longitude": [0.0]
    })

    df_station = df_station.set_index("station_id")

    table_station = client.create_table(
        DB_ID,
        "t_station",
        is_public=False,
        is_schema_public=False,
        dataframe=df_station,
        with_data=False
    )

    print("Created:", table_station.name)

Created: t_station


In [6]:
#################################################
# 6. Create Table: t_time (safe)
#################################################

tables = client.get_tables(DB_ID)
table_names = [t.name for t in tables]

if "t_time" in table_names:
    print("t_time already exists. Skipping.")
else:
    df_time = pd.DataFrame({
        "time_id": [0],
        "date": ["2000-01-01"],
        "hour": [0]
    })

    df_time = df_time.set_index("time_id")

    table_time = client.create_table(
        DB_ID,
        "t_time",
        is_public=False,
        is_schema_public=False,
        dataframe=df_time,
        with_data=False
    )

    print("Created:", table_time.name)

Created: t_time


In [7]:
#################################################
# 7. Create Table: t_measurement (safe)
#################################################

tables = client.get_tables(DB_ID)
table_names = [t.name for t in tables]

if "t_measurement" in table_names:
    print("t_measurement already exists. Skipping.")
else:
    df_measurement = pd.DataFrame({
        "measurement_id": [0],
        "station_id": [0],
        "time_id": [0],

        "SO2": [0.0],
        "NO": [0.0],
        "NO2": [0.0],
        "CO": [0.0],
        "PM10": [0.0],
        "O3": [0.0],
        "PM25": [0.0],

        "wind_direction": [0.0],
        "wind_speed": [0.0],

        "temperature": [0.0],
        "humidity": [0.0],
        "pressure": [0.0],
        "solar_radiation": [0.0],
        "rain": [0.0],

        "BEN": [0.0],
        "TOL": [0.0],
        "MXIL": [0.0]
    })

    df_measurement = df_measurement.set_index("measurement_id")

    table_measurement = client.create_table(
        DB_ID,
        "t_measurement",
        is_public=False,
        is_schema_public=False,
        dataframe=df_measurement,
        with_data=False
    )

    print("Created:", table_measurement.name)

Created: t_measurement


In [8]:
#################################################
# 8. Show DB, Tables, and Columns (FIXED)
#################################################

# Get database name
db_name = None
databases = client.get_databases()

for db in databases:
    if db.id == DB_ID:
        db_name = db.name
        break

print(f"\nTables in DB '{db_name}':\n")

# Get tables
tables = client.get_tables(DB_ID)

for t in tables:
    print(f"\nTable: {t.name}")
    
    table_details = client.get_table(DB_ID, t.id)
    
    # Columns are inside table_details
    for col in table_details.columns:
        print(f"  - {col.name} ({col.type})")


Tables in DB 'data_stew_grp22_air_quality':


Table: t_measurement
  - measurement_id (ColumnType.BIGINT)
  - station_id (ColumnType.BIGINT)
  - time_id (ColumnType.BIGINT)
  - SO2 (ColumnType.DECIMAL)
  - NO (ColumnType.DECIMAL)
  - NO2 (ColumnType.DECIMAL)
  - CO (ColumnType.DECIMAL)
  - PM10 (ColumnType.DECIMAL)
  - O3 (ColumnType.DECIMAL)
  - PM25 (ColumnType.DECIMAL)
  - wind_direction (ColumnType.DECIMAL)
  - wind_speed (ColumnType.DECIMAL)
  - temperature (ColumnType.DECIMAL)
  - humidity (ColumnType.DECIMAL)
  - pressure (ColumnType.DECIMAL)
  - solar_radiation (ColumnType.DECIMAL)
  - rain (ColumnType.DECIMAL)
  - BEN (ColumnType.DECIMAL)
  - TOL (ColumnType.DECIMAL)
  - MXIL (ColumnType.DECIMAL)

Table: t_time
  - time_id (ColumnType.BIGINT)
  - date (ColumnType.DATE)
  - hour (ColumnType.BOOL)

Table: t_station
  - station_id (ColumnType.BIGINT)
  - station_name (ColumnType.VARCHAR)
  - latitude (ColumnType.DECIMAL)
  - longitude (ColumnType.DECIMAL)


In [41]:
#################################################
# 9. Create PID
#################################################

from dbrepo.api.dto import (
    IdentifierType,
    CreateIdentifierCreator,
    CreateIdentifierTitle,
    CreateIdentifierDescription
)

identifier = client.create_identifier(
    DB_ID,
    type=IdentifierType.DATABASE,

    creators=[
        CreateIdentifierCreator(
            creator_name="Ayuntamiento de Gijón",
            name_type="Organizational"
        ),
        CreateIdentifierCreator(
            creator_name="Group 22, TU Wien",
            name_type="Organizational"
        )
    ],

    titles=[
        CreateIdentifierTitle(
            title="Air Quality Dataset – Gijón (2013)"
        )
    ],

    descriptions=[
        CreateIdentifierDescription(
            description="Dataset containing air quality measurements in Gijón, Spain (2013), including pollutants such as SO2, NO, NO2, CO, PM10, PM2.5, O3 and meteorological variables, along with meteorological data.",
            type="Abstract"
        )
    ],

    licenses=[
        {
            "identifier": "CC-BY-3.0-ES",
            "uri": "https://creativecommons.org/licenses/by/3.0/es/",
            "description": "Creative Commons Attribution 3.0 Spain"
        }
    ],

    publisher="Ayuntamiento de Gijón",
    publication_year=2013
)

print("PID created:", identifier.id)

MalformedError: Failed to create identifier: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}

In [14]:
print("Logged in as:", client.whoami())
print("DB owner:", db.owner.username)

12450741
Logged in as: 12450741
DB owner: 12450741


In [12]:
#################################################
# List Databases
#################################################

databases = client.get_databases()

for db in databases:
    print(f"ID: {db.id}, Name: {db.name}")

ID: 17de844a-62fc-406f-abb2-fffb22b44d01, Name: meine-datenbank


In [3]:
#################################################
# List Tables in Database (DBRepo Client)
#################################################

DB_ID = "17de844a-62fc-406f-abb2-fffb22b44d01"

tables = client.get_tables(DB_ID)

print("\nTables in Database:\n")

if len(tables) == 0:
    print("No tables found.")
else:
    for i, t in enumerate(tables):
        print(f"{i+1}. Name: {t.name} | ID: {t.id}")


Tables in Database:

1. Name: second-three | ID: 5455f09e-5dcb-4701-b2d7-426d925e1c7f
2. Name: second-table | ID: 26a4b09e-aba3-4e1f-ab20-5f7162dd500e
3. Name: first-table | ID: e42ce293-86b9-4183-8032-9d79f512bf8f


In [7]:
#################################################
# Print full table data (DataFrame-safe)
#################################################

tables = client.get_tables(DB_ID)

for t in tables:
    print(f"\n=== Table: {t.name} ===\n")

    data = client.get_table_data(DB_ID, t.id)

    # Correct check for empty DataFrame
    if data.empty:
        print("No data.")
    else:
        print(data)


=== Table: second-three ===

No data.

=== Table: second-table ===

No data.

=== Table: first-table ===

   id  temp
0   1   1.0
1   2   2.0


In [10]:
#################################################
# X. List Containers
#################################################

containers = client.get_containers()

print("\nAvailable Containers:\n")

for i, c in enumerate(containers):
    print(f"{i+1}.")
    print(f"   ID: {c.id}")
    print(f"   Name: {c.name}")
    
    # Engine info (important part of your task)
    if hasattr(c, "engine"):
        print(f"   Engine: {c.engine}")
    
    print()


Available Containers:

1.
   ID: 6cfb3b8e-1792-4e46-871a-f3d103527203
   Name: mariadb-galera:11.3.2

